# Text Extraction Slice

This notebook proves the first `llm_client`-backed raw-text extraction boundary in `onto-canon6`. It uses the real prompt asset and the real `TextExtractionService`, while faking the networked structured LLM call so the proof remains deterministic and inspectable.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

SEARCH_ROOTS = [
    Path.cwd().resolve(),
    Path('~/projects/onto-canon6').expanduser(),
]

PROJECT_ROOT = None
for start in SEARCH_ROOTS:
    candidate = start
    while True:
        if (candidate / "src").exists() and (candidate / "notebooks").exists():
            PROJECT_ROOT = candidate
            break
        if candidate.parent == candidate:
            break
        candidate = candidate.parent
    if PROJECT_ROOT is not None:
        break

if PROJECT_ROOT is None:
    raise RuntimeError(
        "Could not locate onto-canon6 repo root from notebook cwd or ~/projects/onto-canon6"
    )

for candidate_path in (
    PROJECT_ROOT / "src",
    PROJECT_ROOT.parent / "llm_client",
    Path('~/projects/llm_client').expanduser(),
):
    if candidate_path.exists():
        candidate_str = str(candidate_path)
        if candidate_str not in sys.path:
            sys.path.insert(0, candidate_str)

from types import SimpleNamespace
import shutil

from llm_client import render_prompt

from onto_canon6.ontology_runtime import clear_loader_caches
from onto_canon6.pipeline import EvidenceSpan, ReviewService
from onto_canon6.pipeline import text_extraction as extraction_module
from onto_canon6.pipeline.text_extraction import (
    ExtractedCandidate,
    ExtractedFiller,
    TextExtractionResponse,
    TextExtractionService,
)
from onto_canon6.surfaces.review_report import ReviewReportService

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

workspace = repo_root / "var" / "notebook_phase4_text_extraction"
if workspace.exists():
    shutil.rmtree(workspace)
workspace.mkdir(parents=True)

clear_loader_caches()
review_service = ReviewService(
    db_path=workspace / "review.sqlite3",
    overlay_root=workspace / "ontology_overlays",
    default_acceptance_policy="record_only",
)
extractor = TextExtractionService(review_service=review_service)
report_service = ReviewReportService(review_service=review_service)
calls: dict[str, object] = {}

print(extractor.prompt_template.relative_to(repo_root))
print(review_service.store.db_path.relative_to(repo_root))


prompts/extraction/text_to_candidate_assertions.yaml
var/notebook_phase4_text_extraction/review.sqlite3


In [2]:
def fake_get_model(task: str) -> str:
    calls["selection_task"] = task
    return "fake-structured-model"

def fake_call_llm_structured(
    model: str,
    messages: list[dict[str, str]],
    response_model: type[TextExtractionResponse],
    **kwargs: object,
) -> tuple[TextExtractionResponse, object]:
    calls["model"] = model
    calls["messages"] = messages
    calls["kwargs"] = kwargs
    response = response_model(
        candidates=(
            ExtractedCandidate(
                predicate="oc:uses_system_demo",
                roles={
                    "subject": (
                        ExtractedFiller(
                            kind="entity",
                            entity_id="ent:activity:mission_planning",
                            entity_type="oc:activity",
                            name="Mission planning",
                        ),
                    ),
                    "object": (
                        ExtractedFiller(
                            kind="entity",
                            entity_id="ent:system:radar_system",
                            entity_type="oc:system",
                            name="radar system",
                        ),
                    ),
                },
                evidence_spans=(
                    EvidenceSpan(start_char=0, end_char=16, text="Mission planning"),
                    EvidenceSpan(start_char=26, end_char=38, text="radar system"),
                ),
                claim_text="Mission planning uses the radar system.",
            ),
        )
    )
    return response, SimpleNamespace(resolved_model="fake-structured-model")

extraction_module._load_llm_client_api = lambda: extraction_module._LLMClientAPI(
    get_model=fake_get_model,
    render_prompt=render_prompt,
    call_llm_structured=fake_call_llm_structured,
)

In [3]:
source_text = "Mission planning uses the radar system during the exercise."

candidate_imports = extractor.extract_candidate_imports(
    source_text=source_text,
    profile_id="default",
    profile_version="1.0.0",
    submitted_by="analyst:notebook",
    source_ref="text://notebook/phase4/demo",
)

assert len(candidate_imports) == 1
candidate_import = candidate_imports[0]
assert calls["selection_task"] == "extraction"
assert isinstance(calls["kwargs"], dict) and calls["kwargs"]["task"] == "extraction"

{
    "candidate_import": candidate_import.model_dump(),
    "rendered_prompt_tail": calls["messages"][-1]["content"],
    "structured_call_kwargs": calls["kwargs"],
}

{'candidate_import': {'profile': {'profile_id': 'default',
   'profile_version': '1.0.0'},
  'payload': {'predicate': 'oc:uses_system_demo',
   'roles': {'subject': [{'kind': 'entity',
      'entity_id': 'ent:activity:mission_planning',
      'entity_type': 'oc:activity',
      'name': 'Mission planning',
      'alias_ids': []}],
    'object': [{'kind': 'entity',
      'entity_id': 'ent:system:radar_system',
      'entity_type': 'oc:system',
      'name': 'radar system',
      'alias_ids': []}]}},
  'submitted_by': 'analyst:notebook',
  'source_artifact': {'source_kind': 'raw_text',
   'source_ref': 'text://notebook/phase4/demo',
   'source_label': None,
   'source_metadata': {},
   'content_text': 'Mission planning uses the radar system during the exercise.'},
  'evidence_spans': ({'start_char': 0,
    'end_char': 16,
    'text': 'Mission planning'},
   {'start_char': 26, 'end_char': 38, 'text': 'radar system'}),
  'claim_text': 'Mission planning uses the radar system.'},
 'rendered_p

In [4]:
submission_results = extractor.extract_and_submit(
    source_text=source_text,
    profile_id="default",
    profile_version="1.0.0",
    submitted_by="analyst:notebook",
    source_ref="text://notebook/phase4/demo",
)
report = report_service.build_report()

assert len(submission_results) == 1
assert report.summary.total_candidates == 1
assert report.candidates[0].claim_text == "Mission planning uses the radar system."
assert report.candidates[0].evidence_spans[0].text == "Mission planning"

{
    "submission_results": [result.model_dump() for result in submission_results],
    "report_summary": report.summary.model_dump(),
    "candidate_row": report.candidates[0].model_dump(),
}

{'submission_results': [{'candidate': {'candidate_id': 'cand_039786728281431a84147563',
    'profile': {'profile_id': 'default', 'profile_version': '1.0.0'},
    'validation_status': 'valid',
    'review_status': 'pending_review',
    'payload_hash': 'sha256:9a6c2eb190a6a952d12f1783bc574b2d42e2896c3f6efd8feb0bcc960a87c105',
    'payload': {'predicate': 'oc:uses_system_demo',
     'roles': {'object': [{'alias_ids': [],
        'entity_id': 'ent:system:radar_system',
        'entity_type': 'oc:system',
        'kind': 'entity',
        'name': 'radar system'}],
      'subject': [{'alias_ids': [],
        'entity_id': 'ent:activity:mission_planning',
        'entity_type': 'oc:activity',
        'kind': 'entity',
        'name': 'Mission planning'}]}},
    'normalized_payload': {'predicate': 'oc:uses_system_demo',
     'roles': {'object': [{'alias_ids': [],
        'entity_id': 'ent:system:radar_system',
        'entity_type': 'oc:system',
        'kind': 'entity',
        'name': 'radar 